# 10 — Final Model

Notebook tích hợp ba model tốt nhất từ Project 3 và chạy thử một tình huống Smart Parking.

Luồng: **Input → Classification + Regression + Demand Prediction → kết quả**.

Notebook này không train lại model.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np

classification_model_path = '../models/classification/best_classification_model.pkl'
regression_model_path = '../models/regression/best_regression_model.pkl'
demand_model_path = '../models/demand/best_demand_model.pkl'

paths = [
    classification_model_path,
    regression_model_path,
    demand_model_path
]

for path in paths:
    print(
        path,
        "→",
        "OK" if os.path.exists(path) else "THIẾU"
    )

In [ ]:
clf_model = joblib.load(classification_model_path)
reg_model = joblib.load(regression_model_path)
demand_model = joblib.load(demand_model_path)

print("Đã load 3 model thành công.")

print("\nClassification features:")
print(clf_model.feature_names_in_.tolist())

print("\nRegression features:")
print(reg_model.feature_names_in_.tolist())

print("\nDemand features:")
print(demand_model.feature_names_in_.tolist())

In [ ]:
def build_clf_reg_features(
    entry_time,
    vehicle,
    usual_zone,
    is_exam_week,
    rolling_avg,
    hist_overnight
):
    data = {
        'is_exam_week': int(is_exam_week),
        'entry_hour': int(entry_time.hour),
        'entry_minute': int(entry_time.minute),
        'day_of_week_num': int(entry_time.weekday()),
        'is_weekend': int(entry_time.weekday() >= 5),
        'is_morning': int(entry_time.hour < 12),
        'rolling_avg_duration': float(rolling_avg),
        'historical_overnight_count': int(hist_overnight),
        'vehicle_type_Motorbike': int(vehicle == 'Motorbike'),
        'usual_zone_Zone_B': int(usual_zone == 'Zone_B'),
        'usual_zone_Zone_C': int(usual_zone == 'Zone_C'),
        'usual_zone_Zone_D': int(usual_zone == 'Zone_D')
    }

    features = pd.DataFrame([data])

    return features.reindex(
        columns=clf_model.feature_names_in_,
        fill_value=0
    )


def build_demand_features(entry_time, is_exam_week):
    features = pd.DataFrame([{
        'hour': int(entry_time.hour),
        'day_of_week': int(entry_time.weekday()),
        'is_weekend': int(entry_time.weekday() >= 5),
        'is_morning': int(entry_time.hour < 12),
        'is_exam_week': int(is_exam_week)
    }])

    return features.reindex(
        columns=demand_model.feature_names_in_,
        fill_value=0
    )

In [ ]:
def smart_parking_predict(
    student_id,
    entry_time_str,
    vehicle='Motorbike',
    usual_zone='Zone_A',
    is_exam_week=0,
    rolling_avg=300,
    hist_overnight=0
):
    entry_time = pd.to_datetime(entry_time_str)

    clf_features = build_clf_reg_features(
        entry_time,
        vehicle,
        usual_zone,
        is_exam_week,
        rolling_avg,
        hist_overnight
    )

    demand_features = build_demand_features(
        entry_time,
        is_exam_week
    )

    predicted_behavior = clf_model.predict(
        clf_features
    )[0]

    predicted_minutes = float(
        reg_model.predict(clf_features)[0]
    )

    predicted_demand = int(
        round(
            max(
                0,
                demand_model.predict(demand_features)[0]
            )
        )
    )

    predicted_minutes = max(
        1,
        int(round(predicted_minutes))
    )

    estimated_exit = entry_time + pd.Timedelta(
        minutes=predicted_minutes
    )

    zone_map = {
        'Early_Return': 'Khu A',
        'Full_Day': 'Khu B/C',
        'Overnight': 'Khu D',
        'Long_Term': 'Khu D'
    }

    return {
        'student_id': student_id,
        'entry_time': entry_time.strftime(
            '%Y-%m-%d %H:%M:%S'
        ),
        'vehicle_type': vehicle,
        'usual_zone': usual_zone,
        'prediction': {
            'parking_behavior': str(predicted_behavior),
            'duration_minutes': predicted_minutes,
            'estimated_exit': estimated_exit.strftime(
                '%Y-%m-%d %H:%M:%S'
            ),
            'current_demand': predicted_demand,
            'recommended_zone': zone_map.get(
                str(predicted_behavior),
                'Khu B/C'
            )
        }
    }

In [ ]:
result = smart_parking_predict(
    student_id='SV2023_CNTT',
    entry_time_str='2023-11-20 07:15:00',
    vehicle='Motorbike',
    usual_zone='Zone_A',
    is_exam_week=0,
    rolling_avg=300,
    hist_overnight=0
)

print(
    json.dumps(
        result,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
metadata = {
    'classification_features': clf_model.feature_names_in_.tolist(),
    'regression_features': reg_model.feature_names_in_.tolist(),
    'demand_features': demand_model.feature_names_in_.tolist(),
    'classification_model_path': classification_model_path,
    'regression_model_path': regression_model_path,
    'demand_model_path': demand_model_path
}

metadata_path = '../models/final_model_metadata.json'

with open(
    metadata_path,
    'w',
    encoding='utf-8'
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Đã lưu metadata:")
print(metadata_path)

## Kết quả

Nếu notebook chạy đến cuối không có lỗi, Project 3 có đầy đủ dataset, feature engineering, các model tốt nhất, kết quả so sánh và một pipeline thử nghiệm đầu vào → model → kết quả.